# GRIT 0.5B on Google Colab (one A100 80GB)

This notebook runs the pinned `Qwen/Qwen2.5-0.5B-Instruct` policy with the pinned `Qwen/Qwen3Guard-Gen-0.6B` safety reward. It keeps the same task corpus, 1,000 projector contexts, 6,000 KL contexts, top-64 plus tail trust region, GRPO grouping, projected AdamW predictor, and central-difference correction as `main.ipynb`.

Choose **Runtime → Change runtime type → A100 80GB GPU**. The actor and persistent vLLM process share GPU 0; this Colab profile is restricted to one actor and the 0.5B policy. Task/evaluation data and preservation prompts are loaded from the cloned repository; missing frozen-base contexts are generated in cell 4. The projector is built once and then reused from Google Drive, alongside complete checkpoints. Colab GPU time and Drive storage are required; the full 40-step duration is unmeasured.

Run cells in order. Context generation, the first projector build, and the training cell may take a long time. Do not put credentials in the notebook or repository.


## 1. Mount Drive and load the repository with prepared data

Set `REPO_URL` to the GitHub repository containing the code and prepared data. The clone must include task/evaluation data and preservation prompt parquets with their manifests. Cell 4 generates missing model-specific contexts from those prompts without resampling them. If using a Drive copy instead, leave `REPO_URL` empty and place the repository including `data/` at `DRIVE_REPO`. A private GitHub repo must be accessible to the Colab runtime without embedding a token in the URL.

If `/content/GRIT` already exists, it is reused. Update that checkout if the required files were pushed after it was cloned. For a repository using Git LFS, retrieve its LFS files before the input check below.


In [1]:
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, subprocess, sys

drive.mount('/content/drive')
REPO_URL = 'https://github.com/namdeptraivcd/GRIT.git'  # e.g. https://github.com/your-name/GRIT.git
DRIVE_REPO = Path('/content/drive/MyDrive/GRIT-source')
REPO = Path('/content/GRIT')
if not REPO.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
    else:
        assert (DRIVE_REPO / 'scripts/train_grit.py').is_file(), 'Set REPO_URL or copy the repository with data and artifacts to DRIVE_REPO.'
        shutil.copytree(DRIVE_REPO, REPO, ignore=shutil.ignore_patterns('.git', 'checkpoints', 'outputs', '__pycache__'))
assert (REPO / 'grit/model_profiles.py').is_file(), 'Source copy is stale; sync the new small-model support.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
print('Source:', REPO)


Mounted at /content/drive
Source: /content/GRIT


## 2. Install the same PyTorch/vLLM stack and check the GPU

Use a fresh Colab GPU runtime. This installs the repository's Linux/CUDA dependencies; if Colab asks for a runtime restart, restart and rerun the setup cells before checking the prepared inputs.


In [2]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-vllm.txt'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() == 1, 'Select a one-GPU A100 80GB runtime.'
properties = torch.cuda.get_device_properties(0)
assert 'A100' in properties.name and properties.total_memory >= 75_000_000_000, f'Need A100 80GB; got {properties.name}, {properties.total_memory} bytes.'
print(properties.name, round(properties.total_memory / 1e9, 1), 'GB')


NVIDIA A100-SXM4-80GB 85.1 GB


## 3. Configure persistent storage and Hub access

Create `HF_TOKEN` in **Colab Secrets** and grant this notebook access. Give it permission to create/write the private checkpoint repo. If Colab Secrets is unavailable or times out, the next cell falls back to a hidden prompt for this runtime only; the token is not written to the notebook. The repository includes 1,000 held-out prompts sampled deterministically from the pinned PKU-SafeRLHF **test** split. The trainer checks their disjointness from preservation prompts before training; use the existing held-out evaluation split. Training generates and safety-scores one greedy response for every held-out prompt after each 2-step update interval. Input data is read from the repository; generated contexts are saved under `DATA_ROOT`, while projectors and checkpoints are persisted on Drive. Keep the generated contexts when reusing a saved projector; they are validated against its construction corpus. With `save_steps=2`, twenty full checkpoints can require roughly 120 GB just for 0.5B FP32 weights and AdamW state; check your Drive quota before training.


In [3]:
from grit.model_profiles import SMALL
from scripts.preservation_data import preservation_paths

DRIVE_ROOT = Path('/content/drive/MyDrive/GRIT_colab_0p5b')
DATA_ROOT = REPO / 'data'  # Input prompts from GitHub; generated contexts are saved here.
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'  # Build once, then reuse across Colab sessions.
OUTPUT_ROOT = DRIVE_ROOT / 'checkpoints'
EVALUATION_FILE = REPO / 'data/evaluation/pku_saferlhf_test_1000.parquet'
HUB_REPO_ID = 'fushinguyenex/GRIT'  # e.g. your-name/grit-qwen25-0p5b
RUN_ID = 'qwen25-0p5b-grit-v1'  # Keep fixed across Colab reconnects for resume.

from getpass import getpass

HF_TOKEN = os.environ.get('HF_TOKEN', '').strip()
if not HF_TOKEN:
    try:
        HF_TOKEN = (userdata.get('HF_TOKEN') or '').strip()
    except Exception as exc:
        print(f'Colab Secrets unavailable ({type(exc).__name__}); using hidden input instead.')
if not HF_TOKEN:
    HF_TOKEN = getpass('HF_TOKEN (input hidden): ').strip()
assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets and grant notebook access.'
assert HUB_REPO_ID, 'Set HUB_REPO_ID.'
os.environ['HF_TOKEN'] = HF_TOKEN
for directory in (ARTIFACT_ROOT, OUTPUT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)
paths = preservation_paths(DATA_ROOT, ARTIFACT_ROOT, SMALL.policy)
TASK_FILE = DATA_ROOT / 'task/task_train.parquet'
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
print({'policy': SMALL.policy, 'reward': SMALL.safety, 'data': str(DATA_ROOT), 'run': str(OUTPUT_DIR)})


Colab Secrets unavailable (TimeoutException); using hidden input instead.
{'policy': 'Qwen/Qwen2.5-0.5B-Instruct', 'reward': 'Qwen/Qwen3Guard-Gen-0.6B', 'data': '/content/GRIT/data', 'run': '/content/drive/MyDrive/GRIT_colab_0p5b/checkpoints/qwen25-0p5b-grit-v1'}


## 4. Generate frozen-base contexts and build the projector

The repository includes prompt-only parquets for both the 1,000 projector and 6,000 KL corpora. Model-specific contexts (`preserve_contexts.parquet`) are generated by forwarding the 0.5B model through every prompt and stored on **Drive** so they survive Colab disconnects.

On the first run, generation takes roughly 2–4 hours on a single A100. Subsequent sessions restore the contexts from Drive and skip straight to the projector check.


In [4]:
import re, shutil

model_key = re.sub(r'[^a-zA-Z0-9._-]', '--', SMALL.policy).lower()
PROMPT_ROOT = DATA_ROOT / 'preservation/split_v1/prompts'
DRIVE_CTX_ROOT = DRIVE_ROOT / 'contexts' / 'split_v1'
REPO_MODEL_ROOT = DATA_ROOT / 'preservation/split_v1' / model_key

for role in ('projector', 'monitor'):
    prompts_file = PROMPT_ROOT / role / 'preserve_prompts.parquet'
    assert prompts_file.is_file(), f'Missing prompt corpus: {prompts_file}'
    drive_ctx_dir = DRIVE_CTX_ROOT / role
    drive_ctx_file = drive_ctx_dir / 'preserve_contexts.parquet'
    repo_ctx_dir = REPO_MODEL_ROOT / role

    if drive_ctx_file.is_file() and drive_ctx_file.stat().st_size > 0:
        # Restore from Drive into the repo tree.
        print(f'Restoring {role} contexts from Drive: {drive_ctx_dir}')
        if repo_ctx_dir.is_symlink() or repo_ctx_dir.exists():
            if repo_ctx_dir.is_symlink():
                repo_ctx_dir.unlink()
            else:
                shutil.rmtree(repo_ctx_dir)
        repo_ctx_dir.parent.mkdir(parents=True, exist_ok=True)
        repo_ctx_dir.symlink_to(drive_ctx_dir)
    else:
        # Generate contexts into repo tree, then copy to Drive.
        print(f'Generating {role} contexts (first run)...', flush=True)
        if repo_ctx_dir.is_symlink():
            repo_ctx_dir.unlink()
        elif repo_ctx_dir.exists():
            shutil.rmtree(repo_ctx_dir)
        subprocess.run([
            sys.executable, '-u', 'scripts/prepare_preservation_data.py', 'generate',
            '--prompts', str(prompts_file),
            '--output-dir', str(repo_ctx_dir),
            '--model-path', SMALL.policy,
            '--revision', SMALL.policy_revision,
            '--max-prompt-length', '2048',
            '--max-new-tokens', '256',
            '--top-k', '64',
            '--dtype', 'float16',
            '--device', 'cuda',
        ], check=True)
        assert (repo_ctx_dir / 'preserve_contexts.parquet').is_file()
        # Persist to Drive.
        drive_ctx_dir.mkdir(parents=True, exist_ok=True)
        for src in repo_ctx_dir.iterdir():
            shutil.copy2(src, drive_ctx_dir / src.name)
        print(f'Saved {role} contexts to Drive: {drive_ctx_dir}')
        # Replace local dir with symlink to Drive.
        shutil.rmtree(repo_ctx_dir)
        repo_ctx_dir.symlink_to(drive_ctx_dir)

# Refresh paths and verify.
paths = preservation_paths(DATA_ROOT, ARTIFACT_ROOT, SMALL.policy)
required_data_inputs = {
    'task dataset': TASK_FILE,
    'held-out evaluation dataset': EVALUATION_FILE,
    'frozen-base projector contexts': Path(paths['projector_contexts']),
    'projector-context manifest': Path(paths['projector_contexts']).parent / 'manifest.json',
    'frozen-base KL contexts': Path(paths['preserve_file']),
    'KL manifest': Path(paths['preserve_file']).parent / 'manifest.json',
}
missing = [f'{name}: {path}' for name, path in required_data_inputs.items()
           if not path.is_file() or path.stat().st_size == 0]
if missing:
    raise FileNotFoundError(
        'Prepared inputs still missing after generation:\n' + '\n'.join(missing))
for name, path in required_data_inputs.items():
    print(f'✓ {name}: {path}')

# Build or reuse projector (already on Drive via ARTIFACT_ROOT).
projector_path = Path(paths['projectors_path'])
if projector_path.is_file() and projector_path.stat().st_size > 0:
    print(f'Reusing projector from Drive: {projector_path}')
else:
    print(f'Building projector at: {projector_path}', flush=True)
    prep_env = dict(os.environ, DATA_ROOT=str(DATA_ROOT), ARTIFACT_ROOT=str(ARTIFACT_ROOT),
                    MODEL_PATH=SMALL.policy, MODEL_REVISION=SMALL.policy_revision,
                    PROJECTORS_PATH=str(projector_path), PYTHON_BIN=sys.executable,
                    PYTHONPATH=str(REPO))
    subprocess.run(['bash', 'scripts/run_preservation_projectors.sh', 'build'],
                   env=prep_env, check=True)
assert projector_path.is_file() and projector_path.stat().st_size > 0, projector_path
print(f'✓ Projector ready: {projector_path}')


Generating projector contexts (first run)...


CalledProcessError: Command '['/usr/bin/python3', '-u', 'scripts/prepare_preservation_data.py', 'generate', '--prompts', '/content/GRIT/data/preservation/split_v1/prompts/projector/preserve_prompts.parquet', '--output-dir', '/content/GRIT/data/preservation/split_v1/qwen--qwen2.5-0.5b-instruct/projector', '--model-path', 'Qwen/Qwen2.5-0.5B-Instruct', '--revision', '7ae557604adf67be50417f59c2c2f167def9a775', '--max-prompt-length', '2048', '--max-new-tokens', '256', '--top-k', '64', '--dtype', 'float16', '--device', 'cuda']' returned non-zero exit status 1.

## 5. Verify CPU invariants and configure training

The global batch, optimizer, KL support, curvature settings and frozen data match `main.ipynb`. One actor handles all 321 prompts per step. `rollout_memory_utilization=0.2` reserves a smaller GPU share for vLLM because it shares the A100 with the actor; this needs a real GPU fit check.


In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'test_function/test_training.py', 'test_function/test_notebook.py'], check=True)
config = {
    'task_file': str(TASK_FILE),
    'preserve_file': paths['preserve_file'],
    'evaluation_file': str(EVALUATION_FILE),
    'projectors_path': paths['projectors_path'],
    'output_dir': str(OUTPUT_DIR),
    'model_path': SMALL.policy,
    'model_revision': SMALL.policy_revision,
    'safety_model_path': SMALL.safety,
    'safety_model_revision': SMALL.safety_revision,
    'rollout_gpu': '0',
    'allow_colocated_rollout': True,
    'rollout_memory_utilization': 0.2,
    'rollout_dtype': 'auto',
    'task_optimizer': 'adamw',
    'projector_relaxation': 0.05,
    'task_batch_size': 250,
    'preserve_batch_size': 48,
    'generations': 5,
    'max_prompt_length': 512,
    'max_response_length': 256,
    'max_preserve_length': 2304,
    'lr': 1e-6,
    'lambda_pres': 1.0,
    'epsilon_pres': 1e-3,
    'top_k': 64,
    'base_statistics_dtype': 'float16',
    'use_curvature': True,
    'fd_radius': 0.05,
    'fd_check_interval': 10,
    'gradient_topk': 5,
    'gradient_checkpointing': True,
    'max_steps': 40,
    'save_steps': 5,
    'validation_steps': 5,
    'hub_repo_id': HUB_REPO_ID,
    'seed': 66,
}
from scripts.train_grit import parse_args
arguments = []
for key, value in config.items():
    flag = '--' + key.replace('_', '-')
    if isinstance(value, bool):
        if value:
            arguments.append(flag)
    elif value is not None:
        arguments.extend([flag, str(value)])
parse_args(arguments)
print('Training configuration validated; output:', OUTPUT_DIR)


## 6. Train or resume from the latest complete Drive checkpoint

This cell starts the GPU training job. Checkpoints are saved every two steps. If Colab ends, reconnect, rerun the setup cells, and execute this cell again; it selects the latest checkpoint with `complete.json`. Keep `RUN_ID` and config unchanged. An interrupted checkpoint directory is renamed and retained so the next save can use its step number. If a run folder exists without any complete checkpoint, inspect it and use a new `RUN_ID`; no data is deleted automatically.


In [ ]:
from datetime import datetime, timezone
if OUTPUT_DIR.exists():
    for candidate in sorted(OUTPUT_DIR.glob('step_*')):
        if candidate.is_dir() and not (candidate / 'complete.json').is_file():
            stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
            archived = OUTPUT_DIR / f'interrupted_{candidate.name}_{stamp}'
            candidate.rename(archived)
            print('Preserved interrupted checkpoint:', archived)
completed = sorted(OUTPUT_DIR.glob('step_*/complete.json'),
                   key=lambda path: int(path.parent.name.split('_')[-1])) if OUTPUT_DIR.exists() else []
train_args = arguments.copy()
if completed:
    last = completed[-1].parent
    last_step = int(last.name.split('_')[-1])
    if last_step >= config['max_steps']:
        print('Training already complete at step', last_step)
    else:
        train_args.extend(['--resume', str(last)])
        print('Resuming from', last)
elif OUTPUT_DIR.exists():
    raise RuntimeError(f'Incomplete run folder without a complete checkpoint: {OUTPUT_DIR}')
if not completed or int(completed[-1].parent.name.split('_')[-1]) < config['max_steps']:
    env = dict(os.environ, CUDA_VISIBLE_DEVICES='0', PYTHONPATH=str(REPO), OMP_NUM_THREADS='4')
    subprocess.run([sys.executable, 'scripts/train_grit.py', *train_args], env=env, check=True)


## 7. Inspect persisted results

`metrics.jsonl`, `diagnostics_summary.json`, complete checkpoints and optimizer state stay in Drive. The first step records KL at the frozen base against the stored top-64 statistics.


In [ ]:
import json
metrics_path = OUTPUT_DIR / 'metrics.jsonl'
if metrics_path.is_file():
    metrics = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
    print('Completed steps:', len(metrics))
    print(json.dumps(metrics[-1], indent=2) if metrics else 'No completed step yet')
summary_path = OUTPUT_DIR / 'diagnostics_summary.json'
if summary_path.is_file():
    print('Diagnostics:', summary_path.read_text())
